In [ ]:
# Environment Setup + Hardware Check
import sys, torch
print("Python:", sys.version)
print("CUDA available:", torch.cuda.is_available())
print("GPU(s):", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

!pip install -U \
    "numpy<2.0" \
    "torch~=2.3.0" \
    "torchvision~=0.18.0" \
    "matplotlib>=3.8,<3.9" \
    ultralytics \
    opencv-python \
    wandb \
    "Pillow==10.3.0" \
    --force-reinstall

print("✅ Đã cài đặt xong các gói tương thích.")

Python: 3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]
CUDA available: True
GPU(s): 2 ['Tesla T4', 'Tesla T4']
  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached torch-2.3.1-cp311-cp311-manylinux1_x86_64.whl.metadata (26 kB)
  Using cached torchvision-0.18.1-cp311-cp311-manylinux1_x86_64.whl.metadata (6.6 kB)
  Using cached matplotlib-3.8.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.8 kB)
  Using cached ultralytics-8.3.217-py3-none-any.whl.metadata (37 kB)
  Using cached opencv_python-4.12.0.88-cp37-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (19 kB)
  Using cached wandb-0.22.2-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (10 kB)
  Using cached pillow-10.3.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (9.2 kB)
  Using cached filelock-3.20.0-py3-none-any.whl.metadata (2.1 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)

In [6]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 16.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 96.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 78.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 31.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 85.5 MB/s eta 0:00:00:00:0100:01
  Attempting unins

In [1]:
import numpy as np, torch
print("✅ NumPy version:", np.__version__)
print("✅ Torch version:", torch.__version__)


✅ NumPy version: 1.26.4
✅ Torch version: 2.6.0+cu124


In [2]:
# Kaggle API Configuration
import os, glob, json, shutil, pathlib

# Setup Kaggle API if kaggle.json is attached via "Add data"
kc = pathlib.Path("/root/.config/kaggle")
kc.mkdir(parents=True, exist_ok=True)

matches = glob.glob("/kaggle/input/**/kaggle.json", recursive=True)
if matches:
    shutil.copy(matches[0], kc/"kaggle.json")
    os.chmod(kc/"kaggle.json", 0o600)
    creds = json.load(open(kc/"kaggle.json"))
    os.environ["KAGGLE_USERNAME"] = creds.get("username", "")
    os.environ["KAGGLE_KEY"] = creds.get("key", "")
    print("✅ Kaggle API configured from:", matches[0])
else:
    print("ℹ️ No kaggle.json found in /kaggle/input — OK if dataset is already added as Input.")

✅ Kaggle API configured from: /kaggle/input/car-detection-kaggle/kaggle.json


In [3]:
# Auto-detect YOLO Traffic Signs Dataset (FIXED VERSION)
import os, glob, pathlib, yaml, subprocess

def find_yolo_roots(base_dir: str):
    """Find YOLO dataset roots by looking for train/images or images/train patterns"""
    roots = set()
    # Look for various YOLO dataset structures
    patterns = [
        "**/train/images", "**/images/train", 
        "**/train", "**/images",
        "**/Train/Images", "**/Images/Train"
    ]
    
    for pat in patterns:
        for p in pathlib.Path(base_dir).glob(pat):
            if "train" in str(p).lower() or "images" in str(p).lower():
                # Try different parent levels
                for parent_level in [0, 1, 2]:
                    try:
                        candidate = p.parents[parent_level]
                        # Check if this looks like a dataset root
                        if any((candidate / subdir).exists() for subdir in ["train", "images", "Train", "Images"]):
                            roots.add(candidate)
                    except IndexError:
                        continue
    
    return sorted(roots, key=lambda p: len(str(p)))  # shorter paths first

# 1) Search in /kaggle/input for YOLO-structured datasets
candidates = find_yolo_roots("/kaggle/input")

source = None
if candidates:
    DATA_ROOT = str(candidates[0])
    source = "input"
    print(f"✅ Found YOLO dataset in input: {DATA_ROOT}")
else:
    # 2) Download via Kaggle API if not found in input
    print("📥 No YOLO-structured dataset found in /kaggle/input → downloading via Kaggle API...")
    work = pathlib.Path("/kaggle/working")
    os.chdir(work)
    dst = work/"data/traffic_signs"
    dst.mkdir(parents=True, exist_ok=True)
    
    # Download Vietnamese traffic signs dataset
    r = subprocess.run([
        "kaggle", "datasets", "download", "-d", 
        "jaydenguyenx/vietnamese-traffic-signs-detection-and-recognition", "-p", "/kaggle/working"
    ], check=False)
    
    if r.returncode != 0:
        raise SystemExit("❌ No YOLO dataset in /kaggle/input and API download failed. Please add dataset: 'jaydenguyenx/vietnamese-traffic-signs-detection-and-recognition'.")
    
    # Extract dataset
    subprocess.run([
        "bash", "-lc", 
        "unzip -o /kaggle/working/vietnamese-traffic-signs-detection-and-recognition.zip -d /kaggle/working/data/traffic_signs >/dev/null"
    ], check=False)
    
    pathlib.Path("/kaggle/working/vietnamese-traffic-signs-detection-and-recognition.zip").unlink(missing_ok=True)
    
    # Debug: List extracted contents
    print("📂 Extracted contents:")
    for root, dirs, files in os.walk("/kaggle/working/data/traffic_signs"):
        level = root.replace("/kaggle/working/data/traffic_signs", "").count(os.sep)
        indent = " " * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        subindent = " " * 2 * (level + 1)
        for file in files[:5]:  # Show first 5 files
            print(f"{subindent}{file}")
        if len(files) > 5:
            print(f"{subindent}... and {len(files)-5} more files")
        if level > 3:  # Limit depth
            break
    
    candidates = find_yolo_roots("/kaggle/working/data/traffic_signs")
    
    if not candidates:
        # Fallback: try to find any directory with images
        print("🔍 No standard YOLO structure found, searching for any image directories...")
        for root, dirs, files in os.walk("/kaggle/working/data/traffic_signs"):
            if any(f.lower().endswith(('.jpg', '.jpeg', '.png')) for f in files):
                candidates.append(pathlib.Path(root).parent)
                print(f"📁 Found images in: {root}")
                break
    
    if not candidates:
        # Last resort: use the extraction directory itself
        candidates = [pathlib.Path("/kaggle/working/data/traffic_signs")]
        print("⚠️ Using extraction directory as dataset root")
    
    DATA_ROOT = str(candidates[0])
    source = "downloaded"
    print(f"✅ Downloaded and extracted to: {DATA_ROOT}")

# Detect dataset structure with flexible patterns
def has(p): 
    return os.path.isdir(os.path.join(DATA_ROOT, p))

def find_image_dirs():
    """Find directories containing images"""
    image_dirs = []
    for root, dirs, files in os.walk(DATA_ROOT):
        if any(f.lower().endswith(('.jpg', '.jpeg', '.png')) for f in files):
            rel_path = os.path.relpath(root, DATA_ROOT)
            image_dirs.append(rel_path)
    return image_dirs

print(f"📂 Searching for image directories in: {DATA_ROOT}")
image_dirs = find_image_dirs()
print(f"📁 Found image directories: {image_dirs}")

# Try different naming conventions
train_patterns = ["train/images", "images/train", "train", "Train/Images", "Train"]
val_patterns = ["valid/images", "val/images", "validation/images", "valid", "val", "Valid", "Validation"]
test_patterns = ["test/images", "test", "Test"]

train = None
for pattern in train_patterns:
    if has(pattern) or pattern in image_dirs:
        train = pattern
        break

val = None
for pattern in val_patterns:
    if has(pattern) or pattern in image_dirs:
        val = pattern
        break

test = None
for pattern in test_patterns:
    if has(pattern) or pattern in image_dirs:
        test = pattern
        break

# Fallback logic
if not train and image_dirs:
    train = image_dirs[0]  # Use first found directory
    print(f"⚠️ No standard train directory found, using: {train}")

if not val:
    val = test if test else train
    print(f"⚠️ No validation set found, using: {val}")

if not test:
    test = val
    print(f"⚠️ No test set found, using: {test}")

print(f"📁 Final dataset structure: train={train}, val={val}, test={test}")

# Verify directories exist and contain images
for split_name, split_path in [("train", train), ("val", val), ("test", test)]:
    full_path = os.path.join(DATA_ROOT, split_path)
    if os.path.exists(full_path):
        image_count = len([f for f in os.listdir(full_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        print(f"✅ {split_name}: {image_count} images in {split_path}")
    else:
        print(f"❌ {split_name}: Directory {split_path} not found!")

📥 No YOLO-structured dataset found in /kaggle/input → downloading via Kaggle API...
Dataset URL: https://www.kaggle.com/datasets/jaydenguyenx/vietnamese-traffic-signs-detection-and-recognition
License(s): unknown


100%|██████████| 1.32G/1.32G [00:01<00:00, 1.24GB/s]



📂 Extracted contents:
traffic_signs/
  train_data/
    custom_data.yaml
    classes.txt
    labels/
      val/
        frame_1867.txt
        frame_1884.txt
        frame_1892.txt
        frame_1961.txt
        frame_1927.txt
        ... and 175 more files
      train/
        frame_463.txt
        9291.txt
        frame_286.txt
        frame_671.txt
        frame_225.txt
        ... and 895 more files
      test/
        9291.txt
        frame_15.txt
        8426.txt
        784.txt
        frame_24.txt
        ... and 85 more files
    images/
      val/
        frame_1914.png
        frame_1853.png
        frame_1811.png
        frame_1926.png
        frame_1949.png
        ... and 175 more files
      train/
        frame_420.png
        frame_603.png
        frame_333.png
        frame_204.png
        frame_1671.png
        ... and 895 more files
      test/
        frame_75.png
        frame_48.png
        frame_55.png
        8156.png
        11881.png
        ... and 85 more f

In [4]:
# Create Traffic Signs Dataset Configuration
os.makedirs("configs", exist_ok=True)
DATA_YAML = "configs/traffic_signs_detection.yaml"

# Traffic signs classes (4 main categories)
traffic_signs_classes = [
    "speed_limit",
    "yield",
    "mandatory",
    "other"
]

# Create YAML configuration
config = {
    "path": DATA_ROOT,
    "train": train,
    "val": val,
    "test": test,
    "nc": len(traffic_signs_classes),
    "names": traffic_signs_classes
}

with open(DATA_YAML, "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print("✅ Created configuration file:", DATA_YAML)
print("📄 Configuration content:")
print(open(DATA_YAML).read())
print(f"📊 Source: {source} | Dataset Root: {DATA_ROOT}")

✅ Created configuration file: configs/traffic_signs_detection.yaml
📄 Configuration content:
path: /kaggle/working/data/traffic_signs/train_data
train: images/train
val: images/train
test: images/train
nc: 4
names:
- speed_limit
- yield
- mandatory
- other

📊 Source: downloaded | Dataset Root: /kaggle/working/data/traffic_signs/train_data


In [8]:
# Initialize YOLOv8 Model and Training
from ultralytics import YOLO
import torch

# Load YOLOv8 nano model (fastest for traffic signs)
model = YOLO("yolov8n.pt")

print(f"🚀 Model loaded: {model.model}")
print(f"🔧 Available devices: {torch.cuda.device_count()} GPUs")

# Training configuration optimized for traffic signs
training_args = {
    # Dataset
    "data": DATA_YAML,
    
    # Training parameters
    "epochs": 50,  # More epochs for smaller objects
    "batch": 32,  # 16 per GPU with 2 GPUs
    "imgsz": 640,
    "device": [0, 1],  # Use both GPUs
    "workers": 4,
    
    # Optimizer (AdamW is better than SGD for detection)
    "optimizer": "AdamW",
    "lr0": 0.001,  # Lower learning rate for stability
    "lrf": 0.01,  # Final learning rate (cosine annealing)
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "warmup_epochs": 3.0,
    "warmup_momentum": 0.8,
    "warmup_bias_lr": 0.1,
    
    # Data augmentation (tuned for traffic signs)
    "hsv_h": 0.01,  # Reduced hue variation (signs have specific colors)
    "hsv_s": 0.5,   # Moderate saturation
    "hsv_v": 0.3,   # Moderate value
    "degrees": 5.0,  # Small rotation (signs are usually upright)
    "translate": 0.05,  # Minimal translation
    "scale": 0.3,   # Scale variation
    "fliplr": 0.5,  # Horizontal flip
    "mosaic": 0.8,  # Mosaic augmentation
    "mixup": 0.1,  # Mixup augmentation
    
    # Output
    "project": "runs_detect",
    "name": "traffic_signs_detection",
    "save": True,
    "save_period": 10,  # Save every 10 epochs
    "cache": True,  # Cache images for faster training
    "amp": True,  # Mixed precision training
    "verbose": True
}

print("🎯 Training configuration:")
for key, value in training_args.items():
    print(f"  {key}: {value}")

🚀 Model loaded: DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C2f(
      (cv1): Conv(
        (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=Tr

In [9]:
# Start Training
print("🚀 Starting YOLOv8 Traffic Signs Detection Training...")
print(f"📊 Dataset: {DATA_ROOT}")
print(f"🎯 Classes: {len(traffic_signs_classes)} - {traffic_signs_classes}")
print(f"⚡ GPUs: {torch.cuda.device_count()}x {torch.cuda.get_device_name(0)}")
print(f"📈 Epochs: {training_args['epochs']}, Batch: {training_args['batch']}")

# Train the model
results = model.train(**training_args)

print("✅ Training completed!")
print(f"📁 Results saved to: {results.save_dir}")

🚀 Starting YOLOv8 Traffic Signs Detection Training...
📊 Dataset: /kaggle/working/data/traffic_signs/train_data
🎯 Classes: 4 - ['speed_limit', 'yield', 'mandatory', 'other']
⚡ GPUs: 2x Tesla T4
📈 Epochs: 50, Batch: 32
Ultralytics 8.3.217 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
                                                       CUDA:1 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=configs/traffic_signs_detection.yaml, degrees=5.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01, hsv_s=0.5, hsv_v=0.3, imgsz=640, int8=False, iou=0.7, keras=False, 

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        124        139      0.856      0.701      0.798      0.558
           speed_limit         23         23       0.71      0.522      0.555      0.366
                 yield         50         54      0.971      0.625      0.834      0.549
             mandatory         21         21      0.741      0.952      0.951      0.734
                 other         41         41          1      0.706      0.853      0.581
Speed: 0.1ms preprocess, 0.8ms inference, 0.0ms loss, 2.0ms postprocess per image
Results saved to /kaggle/working/runs_detect/traffic_signs_detection
✅ Training completed!


AttributeError: 'NoneType' object has no attribute 'save_dir'

In [10]:
import shutil

out_dir = "runs_detect/traffic_signs_detection"
zip_path = "traffic_signs_detection.zip"

# Nén toàn bộ folder thành zip
shutil.make_archive("traffic_signs_detection", 'zip', out_dir)

print(f"✅ Created zip: {zip_path}")


✅ Created zip: traffic_signs_detection.zip
